# SynthID Text: Watermarking for Generated Text

This notebook demonstrates how to use the [SynthID Text library][synthid-code]
to apply and detect watermarks on generated text. It is divided into three major
sections and intended to be run end-to-end.

1.  **_Setup_**: Importing the SynthID Text library, choosing your model (either
    [Gemma 3 270M IT][gemma] or [GPT-2][gpt2]) and device (either CPU or GPU,
    depending on your runtime), defining the watermarking configuration, and
    initializing some helper functions.
1.  **_Applying a watermark_**: Loading your selected model using the
    [Hugging Face Transformers][transformers] library, using that model to
    generate some watermarked text, and comparing the perplexity of the
    watermarked text to that of text generated by the base model.
1.  **_Detecting a watermark_**: Training a detector to recognize text generated
    with a specific watermarking configuration, and then using that detector to
    predict whether a set of examples were generated with that configuration.

As the reference implementation for the
[SynthID Text paper in _Nature_][synthid-paper], this library and notebook are
intended for research review and reproduction only. They should not be used in
production systems.

[gemma]: https://ai.google.dev/gemma/docs/model_card
[gpt2]: https://huggingface.co/openai-community/gpt2
[synthid-code]: https://github.com/google-deepmind/synthid-text
[synthid-paper]: https://doi.org/10.1038/s41586-024-08025-4
[transformers]: https://huggingface.co/docs/transformers/en/index

# 1. Setup

In [ ]:
# @title Install and import the required Python packages
#
# @markdown Running this cell may require you to restart your session.

! pip install 'synthid-text[notebook]>=0.3.0'

from collections.abc import Sequence
import enum
import gc

import datasets
import huggingface_hub
import numpy as np
from synthid_text import detector_bayesian
from synthid_text import detector_mean
from synthid_text import logits_processing
from synthid_text import synthid_mixin
import torch
import tqdm
import transformers

In [ ]:
# @title Choose your model.
#
# @markdown This reference implementation is configured to use Gemma 3 270M IT
# @markdown or GPT-2. Gemma 3 requires a Hugging Face account that has accepted
# @markdown the Gemma license; GPT-2 does not.


class ModelName(enum.Enum):
  GEMMA3_270M = 'google/gemma-3-270m-it'
  GPT2 = 'gpt2'


model_name = 'google/gemma-3-270m-it'  # @param ['google/gemma-3-270m-it', 'gpt2']
MODEL_NAME = ModelName(model_name)

if MODEL_NAME is not ModelName.GPT2:
  huggingface_hub.notebook_login()

In [ ]:
# @title Configure your device
#
# @markdown This notebook loads models from Hugging Face Transformers into the
# @markdown PyTorch deep learning runtime. PyTorch supports generation on CPU or
# @markdown GPU, but your chosen model will run best on the following hardware:
# @markdown
# @markdown * Gemma 3 270M IT: Any GPU will work, such as a T4.
# @markdown * GPT-2: Any runtime will work, though a High-RAM CPU or any GPU
# @markdown   will be faster.

DEVICE = (
    torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
)

if not torch.cuda.is_available() and torch.get_num_threads() > 4:
  torch.set_num_threads(4)

print(f'Using device: {DEVICE} (CPU threads: {torch.get_num_threads()})')
DEVICE

In [ ]:
# @title Example watermarking config
#
# @markdown SynthID Text produces unique watermarks given a configuration, with
# @markdown the most important pieces of a configuration being the `key`: a
# @markdown secret integer in [0, 2**63), and `depth`: the number of layers.
# @markdown
# @markdown This reference implementation uses a fixed watermarking
# @markdown configuration, which will be displayed when you run this cell.

CONFIG = synthid_mixin.DEFAULT_WATERMARKING_CONFIG
CONFIG

In [ ]:
# @title Initialize the required constants, tokenizer, and logits processor

BATCH_SIZE = 8
# 13 batches of 8 give 104 generations, enough to compare perplexities.
NUM_BATCHES = 13
OUTPUTS_LEN = 512
TEMPERATURE = 0.5
TOP_K = 40
TOP_P = 0.99
# GPT-2 degenerates into repetitive text without a repetition penalty. The
# penalty is applied equally to watermarked and unwatermarked generation.
if MODEL_NAME == ModelName.GPT2:
  REPETITION_PENALTY = 1.2
  NO_REPEAT_NGRAM_SIZE = 3
else:
  REPETITION_PENALTY = 1.0
  NO_REPEAT_NGRAM_SIZE = 0

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME.value)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
# Stop token(s) for generation and detection (e.g. [1, 106] for Gemma 3).
EOS_TOKEN_ID = transformers.GenerationConfig.from_pretrained(
    MODEL_NAME.value
).eos_token_id

# Used for detection (g-values and masks). Watermarked models build their own
# processor inside `generate()`, applied after temperature, top-k and top-p.
logits_processor = logits_processing.SynthIDLogitsProcessor(**CONFIG)

In [ ]:
# @title Utility functions to load models, compute perplexity, and process prompts.


def load_model(
    model_name: ModelName,
    expected_device: torch.device,
    enable_watermarking: bool = False,
) -> transformers.PreTrainedModel:
  if model_name == ModelName.GPT2:
    model_cls = (
        synthid_mixin.SynthIDGPT2LMHeadModel
        if enable_watermarking
        else transformers.GPT2LMHeadModel
    )
    model = model_cls.from_pretrained(model_name.value, device_map='auto')
    model.generation_config.pad_token_id = model.generation_config.eos_token_id
  else:
    model_cls = (
        synthid_mixin.SynthIDGemma3ForCausalLM
        if enable_watermarking
        else transformers.Gemma3ForCausalLM
    )
    model = model_cls.from_pretrained(
        model_name.value,
        device_map='auto',
        dtype=torch.bfloat16,
    )
    # Pad finished rows with the tokenizer's EOS token, so that the EOS mask
    # used for perplexity and detection also covers the padding, whichever
    # stop token ended the row.
    model.generation_config.pad_token_id = tokenizer.eos_token_id

  if str(model.device) != str(expected_device):
    raise ValueError('Model device not as expected.')

  return model


def _compute_perplexity(
    outputs: torch.LongTensor,
    logits: Sequence[torch.FloatTensor],
    eos_token_mask: torch.BoolTensor,
) -> tuple[float, int]:
  """Total NLL and token count of the generated tokens.

  The NLL is computed under the model's raw next-token distribution
  (temperature 1, no top-k/top-p), so watermarked and unwatermarked text are
  scored in exactly the same way.
  """
  generated = outputs[:, -len(logits) :]
  log_probs = torch.log_softmax(torch.stack(logits, dim=1).float(), dim=-1)
  nll = -torch.gather(log_probs, 2, generated[..., None]).squeeze(-1)
  total_nll = float((nll * eos_token_mask).sum().item())
  total_tokens = int(eos_token_mask.sum().item())
  return total_nll, total_tokens


def _process_raw_prompt(prompt: Sequence[str]) -> str:
  """Add chat template to the raw prompt."""
  if MODEL_NAME == ModelName.GPT2:
    return prompt.decode().strip('"')
  else:
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt.decode().strip('"')}],
        tokenize=False,
        add_generation_prompt=True,
    )

# 2. Applying a watermark

In [ ]:
# @title Generate watermarked output

gc.collect()
torch.cuda.empty_cache()

batch_size = 1
example_inputs = [
    'I enjoy walking with my cute dog',
    'I am from New York',
    'The test was not so very hard after all',
    "I don't think they can score twice in so short a time",
]
example_inputs = example_inputs * (int(batch_size / 4) + 1)
example_inputs = example_inputs[:batch_size]

inputs = tokenizer(
    example_inputs,
    return_tensors='pt',
    padding=True,
).to(DEVICE)

model = load_model(MODEL_NAME, expected_device=DEVICE, enable_watermarking=True)
torch.manual_seed(0)
max_pos = getattr(model.config, 'max_position_embeddings', 1024)
max_len = min(max_pos, inputs['input_ids'].shape[1] + OUTPUTS_LEN)

outputs = model.generate(
    **inputs,
    do_sample=True,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    repetition_penalty=REPETITION_PENALTY,
    no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
    max_length=max_len,
)

print('Output:\n' + 100 * '-')
for output in outputs:
  print(tokenizer.decode(output, skip_special_tokens=True))
  print(100 * '-')

del inputs, outputs, model
gc.collect()
torch.cuda.empty_cache()

## [Optional] Compare perplexity between watermarked and non-watermarked text

Sample [eli5 dataset](https://facebookresearch.github.io/ELI5/) outputs from
watermarked and non-watermarked models and verify that:

* The [perplexity](https://huggingface.co/docs/transformers/en/perplexity) of
  watermarked and non-watermarked text is similar. Both are measured under
  the base model's next-token distribution at temperature 1 without top-k
  or top-p truncation.

$$\text{PPL}(X) = \exp \left\{ {-\frac{1}{t}\sum_i^t \log p_\theta (x_i|x_{<i}) } \right\}$$

In [ ]:
# @title Load Eli5 dataset with HuggingFace datasets.

eli5_prompts = datasets.load_dataset("Pavithree/eli5")

In [ ]:
# @title Non-watermarked output - perplexity
gc.collect()
torch.cuda.empty_cache()

model = load_model(MODEL_NAME, expected_device=DEVICE)
torch.manual_seed(0)

total_nll_unwm = 0.0
total_tokens_unwm = 0

max_pos = getattr(model.config, 'max_position_embeddings', 1024)

for batch_id in tqdm.tqdm(range(NUM_BATCHES)):
  prompts = eli5_prompts['train']['title'][
      batch_id * BATCH_SIZE : (batch_id + 1) * BATCH_SIZE
  ]
  prompts = [_process_raw_prompt(prompt.encode()) for prompt in prompts]
  inputs = tokenizer(
      prompts,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)
  _, inputs_len = inputs['input_ids'].shape
  max_len = min(max_pos, inputs_len + OUTPUTS_LEN)

  outputs = model.generate(
      **inputs,
      do_sample=True,
      max_length=max_len,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
      repetition_penalty=REPETITION_PENALTY,
      no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
      return_dict_in_generate=True,
      output_logits=True,
  )

  logits = outputs.logits
  outputs = outputs.sequences

  eos_token_mask = logits_processor.compute_eos_token_mask(
      input_ids=outputs[:, inputs_len:],
      eos_token_id=EOS_TOKEN_ID,
  )

  batch_nll, batch_tokens = _compute_perplexity(outputs, logits, eos_token_mask)
  total_nll_unwm += batch_nll
  total_tokens_unwm += batch_tokens

  del inputs, prompts, eos_token_mask, outputs, logits

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
if total_tokens_unwm == 0:
  raise ValueError(
      "No unwatermarked tokens were scored; cannot compute perplexity."
  )
final_perplexity = np.exp(total_nll_unwm / total_tokens_unwm)
print(f"Perplexity of unwatermarked model: {final_perplexity:.2f}")

In [ ]:
# @title Watermarked output - perplexity
gc.collect()
torch.cuda.empty_cache()

model = load_model(MODEL_NAME, expected_device=DEVICE, enable_watermarking=True)
torch.manual_seed(0)

total_nll_wm = 0.0
total_tokens_wm = 0

max_pos = getattr(model.config, 'max_position_embeddings', 1024)

for batch_id in tqdm.tqdm(range(NUM_BATCHES)):
  prompts = eli5_prompts['train']['title'][
      batch_id * BATCH_SIZE : (batch_id + 1) * BATCH_SIZE
  ]
  prompts = [_process_raw_prompt(prompt.encode()) for prompt in prompts]
  inputs = tokenizer(
      prompts,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)
  _, inputs_len = inputs['input_ids'].shape
  max_len = min(max_pos, inputs_len + OUTPUTS_LEN)

  outputs = model.generate(
      **inputs,
      do_sample=True,
      max_length=max_len,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
      repetition_penalty=REPETITION_PENALTY,
      no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
      return_dict_in_generate=True,
      output_logits=True,
  )
  logits = outputs.logits
  outputs = outputs.sequences

  # Mask to ignore all tokens after the end-of-sequence token.
  eos_token_mask = logits_processor.compute_eos_token_mask(
      input_ids=outputs[:, inputs_len:],
      eos_token_id=EOS_TOKEN_ID,
  )

  batch_nll, batch_tokens = _compute_perplexity(outputs, logits, eos_token_mask)
  total_nll_wm += batch_nll
  total_tokens_wm += batch_tokens

  del outputs, logits, inputs, prompts, eos_token_mask

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
if total_tokens_wm == 0:
  raise ValueError(
      "No watermarked tokens were scored; cannot compute perplexity."
  )
final_perplexity = np.exp(total_nll_wm / total_tokens_wm)
print(f"Perplexity of watermarked model: {final_perplexity:.2f}")

# 3. Detecting a watermark

To detect the watermark, you have two options:
1.   Use the simple **Mean** scoring function. This can be done quickly and requires no training.
2.   Use the more powerful **Bayesian** scoring function. This requires training and takes more time.

For full explanation of these scoring functions, see the paper and its Supplementary Materials.


In [ ]:
# @title Constants

NUM_NEGATIVES = 10000
POS_BATCH_SIZE = 32
NUM_POS_BATCHES = 313
NEG_BATCH_SIZE = 32
# Truncate outputs to this length for training.
POS_TRUNCATION_LENGTH = 200
NEG_TRUNCATION_LENGTH = 200
# Pad truncated outputs to this length for equal shape across all batches.
# Must be at least the truncation lengths above; anything longer is
# EOS padding that is masked out but still costs preprocessing time.
MAX_PADDED_LENGTH = 200
TEMPERATURE = 1.0

In [ ]:
# @title Generate model responses


def generate_responses(example_inputs, enable_watermarking):
  inputs = tokenizer(
      example_inputs,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)

  gc.collect()
  torch.cuda.empty_cache()

  model = load_model(
      MODEL_NAME,
      expected_device=DEVICE,
      enable_watermarking=enable_watermarking,
  )
  torch.manual_seed(0)
  _, inputs_len = inputs['input_ids'].shape
  max_pos = getattr(model.config, 'max_position_embeddings', 1024)
  max_len = min(max_pos, inputs_len + OUTPUTS_LEN)

  outputs = model.generate(
      **inputs,
      do_sample=True,
      max_length=max_len,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
      repetition_penalty=REPETITION_PENALTY,
      no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
  )

  # The detectors expect only the generated tokens, without the prompt.
  outputs = outputs[:, inputs_len:]

  mode = 'Watermarked' if enable_watermarking else 'Unwatermarked'
  print(f"\n{'=' * 20} {mode} Responses {'=' * 20}")
  for idx, (prompt, out_tokens) in enumerate(zip(example_inputs, outputs)):
    decoded_text = tokenizer.decode(out_tokens, skip_special_tokens=True)
    print(f'[{idx}] Prompt: {prompt}')
    print(f'    Response: {decoded_text}\n')

  return outputs


example_inputs = [
    'I enjoy walking with my cute dog',
    'I am from New York',
    'The test was not so very hard after all',
    "I don't think they can score twice in so short a time",
]

wm_example_outputs = generate_responses(
    example_inputs, enable_watermarking=True
)
uwm_example_outputs = generate_responses(
    example_inputs, enable_watermarking=False
)

## Option 1: Mean detector

In [ ]:
# @title Get Mean detector scores for the generated outputs.

# Watermarked responses tend to have higher Mean scores than unwatermarked
# responses. To classify responses you can set a score threshold, but this will
# depend on the distribution of scores for your use-case and your desired false
# positive / false negative rates.

# The Mean detectors take g-values and a mask of the positions to score, which
# are computed from the generated token ids.
wm_g_values, wm_mask = logits_processor.compute_g_values_and_mask(
    wm_example_outputs, eos_token_id=EOS_TOKEN_ID
)
uwm_g_values, uwm_mask = logits_processor.compute_g_values_and_mask(
    uwm_example_outputs, eos_token_id=EOS_TOKEN_ID
)

wm_mean_scores = detector_mean.mean_score(
    wm_g_values.cpu().numpy(), wm_mask.cpu().numpy()
)
uwm_mean_scores = detector_mean.mean_score(
    uwm_g_values.cpu().numpy(), uwm_mask.cpu().numpy()
)

print('Mean scores for watermarked responses: ', wm_mean_scores)
print('Mean scores for unwatermarked responses: ', uwm_mean_scores)

# You may find that the Weighted Mean scoring function gives better
# classification performance than the Mean scoring function (in particular,
# higher scores for watermarked responses). See the paper for full details.

wm_weighted_mean_scores = detector_mean.weighted_mean_score(
    wm_g_values.cpu().numpy(), wm_mask.cpu().numpy()
)
uwm_weighted_mean_scores = detector_mean.weighted_mean_score(
    uwm_g_values.cpu().numpy(), uwm_mask.cpu().numpy()
)

print(
    'Weighted Mean scores for watermarked responses: ', wm_weighted_mean_scores
)
print(
    'Weighted Mean scores for unwatermarked responses: ',
    uwm_weighted_mean_scores,
)

## Option 2: Bayesian detector

In [ ]:
# @title Generate watermarked samples for training Bayesian detector

gc.collect()
torch.cuda.empty_cache()

model = load_model(MODEL_NAME, expected_device=DEVICE, enable_watermarking=True)
torch.manual_seed(0)

eli5_prompts = datasets.load_dataset('Pavithree/eli5')

wm_outputs = []

for batch_id in tqdm.tqdm(range(NUM_POS_BATCHES)):
  prompts = eli5_prompts['train']['title'][
      batch_id * POS_BATCH_SIZE : (batch_id + 1) * POS_BATCH_SIZE
  ]
  prompts = [_process_raw_prompt(prompt.encode()) for prompt in prompts]
  inputs = tokenizer(
      prompts,
      return_tensors='pt',
      padding=True,
  ).to(DEVICE)
  _, inputs_len = inputs['input_ids'].shape

  outputs = model.generate(
      **inputs,
      do_sample=True,
      # Only the first POS_TRUNCATION_LENGTH tokens are used for training;
      # generating more wastes time and can exceed GPT-2's 1024 positions.
      max_new_tokens=POS_TRUNCATION_LENGTH,
      temperature=TEMPERATURE,
      top_k=TOP_K,
      top_p=TOP_P,
      repetition_penalty=REPETITION_PENALTY,
      no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
  )

  wm_outputs.append(outputs[:, inputs_len:])

  del outputs, inputs, prompts

del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# @title Generate unwatermarked samples for training Bayesian detector

tokenized_uwm_outputs = []
batched = []
# Pad (on the right) or truncate to this length for batching. Training only
# uses the first NEG_TRUNCATION_LENGTH tokens, which is at most this length.
padded_length = MAX_PADDED_LENGTH

answers = []
for ans_dict in eli5_prompts['train']['answers']:
  if ans_dict['text']:
    answers.append(ans_dict['text'][0])
  if len(answers) >= NUM_NEGATIVES:
    break

for text in tqdm.tqdm(answers):
  line = tokenizer(text)['input_ids']
  if len(line) >= padded_length:
    line = line[:padded_length]
  else:
    line = line + [
        tokenizer.eos_token_id for _ in range(padded_length - len(line))
    ]
  batched.append(torch.tensor(line, dtype=torch.long, device=DEVICE)[None, :])
  if len(batched) == NEG_BATCH_SIZE:
    tokenized_uwm_outputs.append(torch.cat(batched, dim=0))
    batched = []

In [ ]:
# @title Train the Bayesian detector
bayesian_detector, test_loss = (
    detector_bayesian.BayesianDetector.train_best_detector(
        tokenized_wm_outputs=wm_outputs,
        tokenized_uwm_outputs=tokenized_uwm_outputs,
        logits_processor=logits_processor,
        tokenizer=tokenizer,
        torch_device=DEVICE,
        max_padded_length=MAX_PADDED_LENGTH,
        pos_truncation_length=POS_TRUNCATION_LENGTH,
        neg_truncation_length=NEG_TRUNCATION_LENGTH,
        verbose=True,
        learning_rate=3e-3,
        n_epochs=100,
        l2_weights=np.zeros((1,)),
    )
)

In [ ]:
# @title Get Bayesian detector scores for the generated outputs.

# Watermarked responses tend to have higher Bayesian scores than unwatermarked
# responses. To classify responses you can set a score threshold, but this will
# depend on the distribution of scores for your use-case and your desired false
# positive / false negative rates. See the paper for full details.

wm_bayesian_scores = bayesian_detector.score(wm_example_outputs)
uwm_bayesian_scores = bayesian_detector.score(uwm_example_outputs)

print('Bayesian scores for watermarked responses: ', wm_bayesian_scores)
print('Bayesian scores for unwatermarked responses: ', uwm_bayesian_scores)